# 03 - Improvements: ensemble, class imbalance, baseline vs improved

Run this after notebook 02. All the code is in `src/improvements.py`; this notebook runs it step by step and explains the results.

| Step | What | Why |
|---|---|---|
| 1 | **Soft-voting ensemble** of Logistic Regression, Random Forest and Gradient Boosting | performance improvement by ensembling |
| 2 | **Class-imbalance experiment** on Framingham (15.2% positive): no treatment vs `class_weight="balanced"` vs SMOTE | check and address class imbalance |
| 3 | **Baseline vs improved** table and chart | show the gain over the baseline in one place |

The rules from notebook 02 still hold: dataset A is the only data used to train and select models for the app, and B (Framingham) is only used for testing.
The imbalance experiment (step 2) is the one exception: it trains inside Framingham's own 5-fold CV (the Exp 3 setting), because A is balanced and has no imbalance to fix.

In [1]:
import sys
import warnings
from pathlib import Path

import pandas as pd
from IPython.display import display

warnings.filterwarnings("ignore")
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
import improvements as im

pd.set_option("display.width", 200, "display.max_columns", 40)
a, b = im.load_data()
print(f"A: {len(a):,} rows, positive rate {a.target.mean():.3f}")
print(f"B: {len(b):,} rows, positive rate {b.target.mean():.3f}")

A: 68,515 rows, positive rate 0.495
B: 4,238 rows, positive rate 0.152


## 1. Soft-voting ensemble (Exp 4)

**How it works:** each of the three models gives a probability, and the ensemble uses their average.
The models make different kinds of mistakes (a linear model vs two tree models), so averaging can smooth out errors.

**How it is tested (no leakage):** the same nested 5-fold CV on A as notebook 02, with the same outer folds.
In each outer fold, each member uses the setting that the inner grid search chose *in that fold* (saved in `exp1_cv_fold_metrics.csv`),
so the outer test fold is never used for tuning. Then the ensemble is trained on all of A and tested on B.

In [2]:
ens = im.run_ensemble(a, b)
print("Ensemble, nested 5-fold CV on A (per fold):")
display(ens["folds"].round(4))
print(f"Mean ROC-AUC {ens['folds'].roc_auc.mean():.4f}, mean recall at 70% specificity {ens['folds'].recall_at_70spec.mean():.4f}")
t = ens["test_B"]
print(f"Trained on A, tested on B: ROC-AUC {t['B_roc_auc']:.3f} (95% CI {t['B_roc_auc_ci_low']:.3f}-{t['B_roc_auc_ci_high']:.3f}), PR-AUC {t['B_pr_auc']:.3f}")

ensemble fold 1: AUC 0.7948, recall@70spec 0.7518 (2s)


ensemble fold 2: AUC 0.7969, recall@70spec 0.7560 (4s)


ensemble fold 3: AUC 0.7986, recall@70spec 0.7566 (6s)


ensemble fold 4: AUC 0.8047, recall@70spec 0.7609 (8s)


ensemble fold 5: AUC 0.8009, recall@70spec 0.7604 (10s)


ensemble trained on all of A; on B: AUC 0.6703 (14s)


ensemble importance done (44s)


Ensemble, nested 5-fold CV on A (per fold):


,model,fold,roc_auc,pr_auc,recall_at_70spec
0,Soft-voting ensemble,1,0.7948,0.7751,0.7518
1,Soft-voting ensemble,2,0.7969,0.7744,0.7560
2,Soft-voting ensemble,3,0.7986,0.7828,0.7566
3,Soft-voting ensemble,4,0.8047,0.7843,0.7609
4,Soft-voting ensemble,5,0.8009,0.7829,0.7604


Mean ROC-AUC 0.7992, mean recall at 70% specificity 0.7571
Trained on A, tested on B: ROC-AUC 0.670 (95% CI 0.650-0.691), PR-AUC 0.249


### Model selection with the ensemble included

Same rule as notebook 02: the highest mean recall at 70% specificity; models within 0.02 of the best count as tied and are ranked by mean ROC-AUC.

In [3]:
sel, best = im.update_selection(ens)
comp = im.save_ensemble(ens, sel, best)
display(sel.round(4))
print("Selected model for the app:", best)

,recall70_mean,recall70_sd,auc_mean,auc_sd,tied_with_best,selected
model,,,,,,
Gradient Boosting,0.7555,0.0049,0.8001,0.0034,True,True
Logistic Regression,0.7473,0.0050,0.7906,0.0037,True,False
Random Forest,0.7573,0.0049,0.7996,0.0037,True,False
Soft-voting ensemble,0.7571,0.0037,0.7992,0.0038,True,False


Selected model for the app: Gradient Boosting


In [4]:
fig = im.plot_roc_with_ensemble(a, b, ens)

## 2. Class-imbalance experiment (Exp 5)

**The problem:** in Framingham only 15.2% of people develop CHD. A model trained on this data learns that "no CHD" is almost always right,
so at the default 0.5 cutoff it hardly ever predicts the positive class: high accuracy, but it misses most of the people we care about.

**Three treatments compared**, each with the same 5-fold CV and inner grid search as Exp 3:
* **none**: train as usual
* **class_weight="balanced"**: mistakes on the rare class cost more during training (about 5.6 times more here)
* **SMOTE**: create synthetic positive examples by interpolating between real ones. It runs inside the training folds only
  (an `imblearn` pipeline), so the test folds are never resampled and no synthetic person is ever tested on.

**What to look at:** recall and F1 at the 0.5 cutoff (the cutoff-dependent measures that imbalance hurts),
and ROC-AUC (ranking quality, which imbalance treatment should not change much).

In [5]:
imb = im.run_imbalance(b)
display(imb.round(3))

imbalance fold 1 done (21s)


imbalance fold 2 done (41s)


imbalance fold 3 done (62s)


imbalance fold 4 done (84s)


imbalance fold 5 done (105s)


,model,strategy,roc_auc,pr_auc,pr_auc_baseline,recall,specificity,precision,f1,balanced_accuracy,flagged_pct
0,Logistic Regression,none,0.723,0.328,0.152,0.051,0.993,0.559,0.094,0.522,1.392
1,Logistic Regression,class_weight,0.724,0.330,0.152,0.675,0.673,0.270,0.386,0.674,38.037
2,Logistic Regression,SMOTE,0.722,0.326,0.152,0.682,0.670,0.270,0.387,0.676,38.344
3,Random Forest,none,0.712,0.312,0.152,0.002,1.000,1.000,0.003,0.501,0.024
4,Random Forest,class_weight,0.710,0.310,0.152,0.641,0.671,0.259,0.369,0.656,37.612
5,Random Forest,SMOTE,0.709,0.312,0.152,0.189,0.946,0.385,0.254,0.568,7.480
6,Gradient Boosting,none,0.705,0.298,0.152,0.045,0.989,0.420,0.081,0.517,1.628
7,Gradient Boosting,class_weight,0.705,0.308,0.152,0.635,0.674,0.259,0.368,0.655,37.282
8,Gradient Boosting,SMOTE,0.709,0.304,0.152,0.138,0.970,0.449,0.211,0.554,4.672


In [6]:
fig = im.plot_imbalance(imb)

## 3. Baseline vs improved

In [7]:
summary, imb_summary, exp3 = im.build_summary(comp, imb)
print("Ranking quality (ROC-AUC), gain over the Logistic Regression baseline")
display(summary.round(3))
print("Imbalance treatment on Framingham, gain over Logistic Regression with no treatment")
display(imb_summary.round(3))

Ranking quality (ROC-AUC), gain over the Logistic Regression baseline


,step,model,A_cv_roc_auc,A_gain_vs_baseline,A_cv_recall_at_70spec,B_roc_auc,B_gain_vs_baseline,B_pr_auc
0,Baseline,Logistic Regression,0.791,0.000,0.747,0.680,0.000,0.280
1,Tuned non-linear model,Random Forest,0.800,0.009,0.757,0.648,-0.031,0.220
2,Tuned non-linear model,Gradient Boosting,0.800,0.010,0.756,0.661,-0.018,0.232
3,Ensembling,Soft-voting ensemble,0.799,0.009,0.757,0.670,-0.009,0.249


Imbalance treatment on Framingham, gain over Logistic Regression with no treatment


,model,strategy,recall_at_0.5,f1_at_0.5,balanced_accuracy,roc_auc,recall_gain_vs_baseline,f1_gain_vs_baseline
0,Logistic Regression,none,0.051,0.094,0.522,0.723,0.000,0.000
1,Logistic Regression,class_weight,0.675,0.386,0.674,0.724,0.624,0.292
2,Logistic Regression,SMOTE,0.682,0.387,0.676,0.722,0.630,0.293
3,Random Forest,none,0.002,0.003,0.501,0.712,-0.050,-0.091
4,Random Forest,class_weight,0.641,0.369,0.656,0.710,0.590,0.275
5,Random Forest,SMOTE,0.189,0.254,0.568,0.709,0.138,0.160
6,Gradient Boosting,none,0.045,0.081,0.517,0.705,-0.006,-0.013
7,Gradient Boosting,class_weight,0.635,0.368,0.655,0.705,0.584,0.274
8,Gradient Boosting,SMOTE,0.138,0.211,0.554,0.709,0.087,0.118


In [8]:
fig = im.plot_summary(summary, imb)

In [9]:
lr = summary.set_index("model").loc["Logistic Regression"]
gb = summary.set_index("model").loc["Gradient Boosting"]
en = summary.set_index("model").loc[im.ENSEMBLE]
i = imb.set_index(["model", "strategy"])
print(f"On A: baseline AUC {lr.A_cv_roc_auc:.3f} -> Gradient Boosting {gb.A_cv_roc_auc:.3f} ({gb.A_gain_vs_baseline:+.3f}), ensemble {en.A_cv_roc_auc:.3f} ({en.A_gain_vs_baseline:+.3f})")
print(f"On B: baseline AUC {lr.B_roc_auc:.3f}; Gradient Boosting {gb.B_roc_auc:.3f}, ensemble {en.B_roc_auc:.3f}")
print(f"Imbalance, Logistic Regression on Framingham at 0.5: recall {i.loc[('Logistic Regression','none')].recall:.2f} -> "
      f"{i.loc[('Logistic Regression','class_weight')].recall:.2f} (class_weight) / {i.loc[('Logistic Regression','SMOTE')].recall:.2f} (SMOTE); "
      f"F1 {i.loc[('Logistic Regression','none')].f1:.2f} -> {i.loc[('Logistic Regression','class_weight')].f1:.2f} / {i.loc[('Logistic Regression','SMOTE')].f1:.2f}")

On A: baseline AUC 0.791 -> Gradient Boosting 0.800 (+0.010), ensemble 0.799 (+0.009)
On B: baseline AUC 0.680; Gradient Boosting 0.661, ensemble 0.670
Imbalance, Logistic Regression on Framingham at 0.5: recall 0.05 -> 0.68 (class_weight) / 0.68 (SMOTE); F1 0.09 -> 0.39 / 0.39


## 4. Findings

* **Within A**, the tuned tree models and the ensemble improve on the Logistic Regression baseline by about 0.01 ROC-AUC.
  The ensemble is level with the best single model, not above it. A has only 8 attributes, so there is little left for averaging to add.
* **On B (a different population)**, the order flips: the simple baseline transfers best, and the tree models lose the most.
  The ensemble sits in between: it loses less than either tree model, so averaging makes the transfer more stable.
  The selection rule still uses A only, so the app's default model is chosen without looking at B.
* **Class imbalance is the biggest improvement.** Without treatment, the models trained on Framingham catch only about 5% of the CHD cases at the 0.5 cutoff.
  With `class_weight="balanced"` they catch about 64-68%, and F1 rises from under 0.1 to about 0.37-0.39.
  SMOTE works as well as class weighting for Logistic Regression but much less well for the tree models, so **class weighting is the better choice here** (simpler, and good for all three).
* ROC-AUC barely changes with any treatment. That is expected: imbalance treatment moves the decision cutoff, not how well the model ranks people.
  Choosing a cutoff on purpose (as the app does with its recall-75 cutoff) is the other way to deal with the same problem.
* SMOTE here runs after one-hot encoding, so the synthetic people can have in-between level values; this is a known limit of plain SMOTE on mixed data.